# Rule Extraction — Holdout (Train/Test Split)

Rule extraction on a stratified train/test split. **Prerequisite:** run
`pipe.run_baseline_split(k_values=...)` (notebook 02, section 2b) first for
every k you want rule extraction on — this notebook no longer derives its
own split/FS; it reuses `outputs_baseline_split/k{k}/{dataset}/` directly.

For each k in `holdout.batch.rare_class_k_values` (a subset of the split-baseline's
own k sweep), for each dataset, this notebook:

1. Reads the split (train/test row indices) and feature-selection artifacts
   (`holdout.fs_methods` — boruta, mrmr_k50, mrmr_k75) already written by
   `run_baseline_split()`.
2. Trains each rule model (`holdout.models` — rf, decisiontree)
   `holdout.n_repeats` times, keeps the best run by `holdout.select_metric`,
   and records **both train- and test-set metrics** of the winner.
3. Mines rules from the winning model, evaluates them on train + a
   generalisation check on test, runs SHAP.
4. Cross-checks rf vs. decisiontree rules; writes a provenance comparison.
5. Exports the held-out test set for later single-sample UI testing.


## Setup

In [ ]:
import sys
import json
import warnings
from pathlib import Path

# Make src.* importable from the notebooks/ directory
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import numpy as np
import pandas as pd
from IPython.display import Image, display

from src.pipeline import GeneExpressionPipeline

pipe = GeneExpressionPipeline(
    config_root=str(REPO_ROOT / "configs"),
    output_root=str(REPO_ROOT / "outputs_baseline_full"),  # unused by this flow; kept for pipe.print_summary()
    use_wandb=False,
)

# outputs_holdout/k{K}/ -- where this notebook's results land, one per k.
HOLDOUT_ROOT = REPO_ROOT / "outputs_holdout"
SPLIT_ROOT = REPO_ROOT / "outputs_baseline_split"


: 

## Run Holdout Rule Extraction

`pipe.run_rule_extraction_holdout(dataset_names=None, k_values=K_VALUES)` —
all enabled datasets, sweeping every k in `K_VALUES`. Each k must already have
`outputs_baseline_split/k{k}/{dataset}/` populated (run notebook 02's section
2b first) — this cell will report and skip any (dataset, k, fs_method) combo
that's missing artifacts rather than fail the whole run.


In [ ]:
holdout_cfg = dict(pipe.config_loader.load_yaml('holdout.yaml').get('holdout', {}))
K_VALUES = holdout_cfg.get('batch', {}).get('rare_class_k_values', [3, 4, 5])
print(f"Rule extraction for k={K_VALUES}")

all_k_results = pipe.run_rule_extraction_holdout(dataset_names=None, k_values=K_VALUES)
all_k_results


## Inspect the Split

Choose which k to inspect below (`INSPECT_K`, defaults to the first of `K_VALUES`).
Rare-class removal (< that k) and the stratified train/test partition are read from
the split-baseline's artifacts (`outputs_baseline_split/k{k}/{dataset}/`) that this
run reused — not re-derived here.


In [ ]:
INSPECT_K = K_VALUES[0]
DATASET = 'GEO-Breast-20711'
ds_root = HOLDOUT_ROOT / f'k{INSPECT_K}' / DATASET
split_root_ds = SPLIT_ROOT / f'k{INSPECT_K}' / DATASET

split_info = json.loads((split_root_ds / 'split_info.json').read_text())
print(json.dumps(split_info, indent=2))


In [ ]:
display(Image(filename=str(split_root_ds / 'visualizations' / 'class_distribution_after_dropna.png')))
display(Image(filename=str(split_root_ds / 'visualizations' / 'train_test_split.png')))


### All datasets — class distribution overview

`run_baseline_split()` already generates this combined grid automatically per k
(`outputs_baseline_split/k{k}/visualizations/`) — just display it, no need to
rebuild it here.


In [ ]:
display(Image(filename=str(SPLIT_ROOT / f'k{INSPECT_K}' / 'visualizations' / 'all_datasets_class_distribution.png')))
display(Image(filename=str(SPLIT_ROOT / f'k{INSPECT_K}' / 'visualizations' / 'all_datasets_metric_comparison.png')))


## Inspect the Held-Out Test Set

Exported once per dataset (raw feature space) — `test_set.csv` for bulk use, `manifest.csv` to
list samples, and one JSON per sample under `samples/` for later single-sample UI testing.

In [ ]:
manifest = pd.read_csv(ds_root / 'test_set' / 'manifest.csv')
manifest

In [ ]:
# One sample, as the UI would load it for a single-sample prediction demo.
sample_id = manifest.iloc[0]['sample_id']
sample_payload = json.loads((ds_root / 'test_set' / 'samples' / f'{sample_id}.json').read_text())
print('sample_id:', sample_payload['sample_id'], '· true_label:', sample_payload['true_label'])
list(sample_payload['features'].items())[:10]

## Inspect Outputs — Random Forest

`cv_results.csv` (one row per repeated-training run), `cv_summary.csv` (mean ± std across runs),
and the rules mined from the winning run.

In [ ]:
FS_METHOD = 'mrmr_k50'
rf_dir = ds_root / FS_METHOD / 'rf'

cv_results = pd.read_csv(rf_dir / 'models' / 'cv_results.csv')
cv_summary = pd.read_csv(rf_dir / 'models' / 'cv_summary.csv')
print('Per-run metrics (n_repeats rows):'); display(cv_results)
print('Mean ± std across runs:'); display(cv_summary)

In [ ]:
# Confusion matrix of the winning run, on the held-out test set.
display(Image(filename=str(rf_dir / 'models' / 'confusion_matrix_test.png')))
pd.read_csv(rf_dir / 'models' / 'confusion_matrix_test.csv', index_col=0)

In [ ]:
rf_rules = pd.read_csv(rf_dir / 'rules' / 'rules.csv')
rf_summary = json.loads((rf_dir / 'rules' / 'rules_summary.json').read_text())
print('RF set-level (train) + best-run train+test metrics:')
print('  test :', rf_summary.get('best_run_test_metrics'))
print('  train:', rf_summary.get('best_run_train_metrics'))
rf_rules.head(10)


In [ ]:
# Generalisation check: the SAME (train-mined) rules re-evaluated on the held-out test set.
rf_test_eval = pd.read_csv(rf_dir / 'rules' / 'rules_test_eval.csv')
rf_test_eval.head(10)

## Inspect Outputs — Decision Tree

In [ ]:
dt_dir = ds_root / FS_METHOD / 'dt'
print((dt_dir / 'rules' / 'rules_human.txt').read_text())
pd.read_csv(dt_dir / 'rules' / 'rules.csv')

## Gene Descriptions — Biological Annotation (separate step)

Enrich the genes appearing in the holdout rules with biological info from each dataset's
platform file (`Platform_Annotations/…`): **Gene title, Entrez ID, GenBank, chromosome,
GO Biological Process / Cellular Component / Molecular Function**. Writes
`gene_description.csv` + `gene_description.md` next to each `genes_in_rules.csv` under
`outputs_holdout/{dataset}/{fs_method}/{rf,dt}/rules/`.

This is a **separate step**: it reads existing rule outputs (no re-training), so it can run
any time after `run_rule_extraction_holdout()`.

In [ ]:
# Separate step -- annotate holdout rule genes from the platform files.
gene_desc = pipe.describe_genes_in_rules_holdout(dataset_names=None, k=INSPECT_K)
gene_desc


In [ ]:
# Preview one dataset's gene descriptions (RF, mrmr_miq)
gene_desc_csv = rf_dir / 'rules' / 'gene_description.csv'
if gene_desc_csv.exists():
    display(pd.read_csv(gene_desc_csv).head(10))
else:
    print(f'Not found: {gene_desc_csv}')

## Cross-Check (rf vs. decisiontree) & Provenance

`DATASET_PROVENANCE_HOLDOUT.md` compares every fs_method × model on `select_metric` (default
`f1_macro`).

In [ ]:
crosscheck = json.loads((ds_root / FS_METHOD / 'compare' / 'crosscheck.json').read_text())
print(json.dumps(crosscheck, indent=2))

In [ ]:
print((ds_root / 'DATASET_PROVENANCE_HOLDOUT.md').read_text())

In [ ]:
for k in [4, 5]:
    print(f"Generating gene descriptions for k={k}")
    pipe.describe_genes_in_rules_holdout(
        dataset_names=None,
        k=k,
    )


: 